In [ ]:

# ==========================================
# LINEAR REGRESSION IMPLEMENTATION(general)
# ==========================================

# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==========================================
# 2. Load dataset
# ==========================================

file_path = "PASTE_YOUR_DATASET_PATH_HERE"
target_column = "PASTE_TARGET_COLUMN_HERE"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print(df.head())

# ==========================================
# 3. Prepare data
# ==========================================

# Remove rows with missing target values
df = df.dropna(subset=[target_column])

# Convert target into numeric format
df[target_column] = pd.to_numeric(
    df[target_column], errors="coerce"
)

df = df.dropna(subset=[target_column])

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]

print("\nFeatures:", X.columns.tolist())
print("Target:", target_column)

# Remove columns containing only missing values
X = X.dropna(axis=1, how="all")

# ==========================================
# 4. Identify numerical and categorical data
# ==========================================

numeric_features = X.select_dtypes(
    include=np.number
).columns

categorical_features = X.select_dtypes(
    exclude=np.number
).columns

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# ==========================================
# 5. Split dataset
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)

# ==========================================
# 6. Create Linear Regression model
# ==========================================

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train model
model.fit(X_train, y_train)

print("\nLinear Regression Training Completed!")

# ==========================================
# 7. Make predictions
# ==========================================

y_pred = model.predict(X_test)

print("\nActual Values:")
print(y_test.head().values)

print("\nPredicted Values:")
print(y_pred[:5])

# ==========================================
# 8. Evaluate model
# ==========================================

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("\nMODEL EVALUATION")
print("-------------------------")
print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("R2 Score:", r2)

# ==========================================
# 9. Actual vs Predicted Graph
# ==========================================

plt.figure(figsize=(8, 5))

plt.scatter(y_test, y_pred)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted Values")

# Ideal prediction line
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

plt.plot([low, high], [low, high], "r--")

plt.grid()
plt.show()

# ==========================================
# 10. Residual Plot
# ==========================================

residuals = y_test - y_pred

plt.figure(figsize=(8, 5))

plt.scatter(y_pred, residuals)
plt.axhline(y=0, color="red", linestyle="--")

plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot")

plt.grid()
plt.show()

# ==========================================
# 11. Display Model Coefficients
# ==========================================

regressor = model.named_steps["regressor"]

feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": regressor.coef_
})

print("\nModel Intercept:", regressor.intercept_)

print("\nFeature Coefficients:")
print(coefficients)

# ==========================================
# 12. Compare Actual and Predicted Values
# ==========================================

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print("\nActual vs Predicted:")
print(comparison.head(10))
